In [1]:
"""
raga_classification_mfcc_chroma.py
-----------------------------------
Classifies ragas using MFCC (timbral) and chroma (pitch-class) features,
mirroring the PCD/PCDD evaluation: reports accuracy for MFCC alone,
chroma alone, and MFCC+chroma combined.

For each of the three feature sets, this runs:
  1. 10-fold cross-validation accuracy (the "in-sample" success rate).
  2. "Unseen" evaluation -- testing on performances/artists that never
     appeared in training at all, which is a much harder, more realistic test.

WHAT YOU NEED
Your dataset is currently split into one CSV per raga (e.g. "madhukauns.csv"),
each with 10 rows (one per song/performance), and columns:
  filepath, artist, raga, tonic_hz,
  mfcc_0_mean..mfcc_12_std, mfcc_delta_0_mean..mfcc_delta_12_std,
  chroma_0_mean..chroma_11_std

Put all your per-raga CSVs into one folder (e.g. 10 files for 10 ragas) and
point DATA_DIR at it below.

HOW TO RUN
    On Kaggle: paste this whole file into one cell, edit DATA_DIR, run.
    Or as a script:  python raga_classification_mfcc_chroma.py --data_dir /path/to/csvs
"""

import os
import glob
import argparse
import unicodedata
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, GroupShuffleSplit, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# ============================================================
# CONFIG -- edit this to match your setup
# ============================================================
DATA_DIR = "/kaggle/input/datasets/mishbhaul/mfcc-chroma-hcm-dataset"   # folder containing one CSV per raga
N_CV_FOLDS = 10
UNSEEN_TEST_SIZE = 0.1   # fraction of ARTISTS (not songs) held out per raga for the unseen test
RANDOM_STATE = 42
FEATURE_SETS = ["mfcc", "chroma", "both"]   # which feature sets to evaluate, in order
# ============================================================


# ===========================================================================
# 1. LOAD AND COMBINE ALL PER-RAGA CSVs INTO ONE DATAFRAME
# ===========================================================================
def load_all_ragas(data_dir: str) -> pd.DataFrame:
    csv_paths = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    if not csv_paths:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    dfs = []
    for path in csv_paths:
        df = pd.read_csv(path)
        dfs.append(df)
        ragas_here = df['raga'].unique().tolist()
        print(f"  loaded {os.path.basename(path)}: {len(df)} rows, raga(s)={ragas_here}")

    combined = pd.concat(dfs, ignore_index=True)
    print(f"\nCombined dataset: {len(combined)} rows, {combined['raga'].nunique()} raga classes")
    return combined


# ===========================================================================
# 1b. PREPROCESSING -- run this on the combined dataframe BEFORE anything else.
#     Small hand-built datasets like this one commonly have quiet issues that
#     silently hurt a 100-row dataset far more than they would a huge one, so
#     each check below is intentionally verbose about what it finds/fixes.
#     Uses ALL feature columns (mfcc_ + chroma_) for these checks regardless
#     of which feature set will actually be modeled later -- a bad row is bad
#     for every feature set, and a column-set-specific check would just
#     repeat this three times for no benefit.
# ===========================================================================
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    print(f"\n{'='*60}\nPREPROCESSING\n{'='*60}")
    df = df.copy()
    feature_cols = get_feature_columns(df, "both")

    # --- 1. Normalize text columns (raga, artist) ---
    for col in ["raga", "artist"]:
        if col in df.columns:
            before = df[col].nunique()
            df[col] = df[col].astype(str).str.strip().apply(lambda s: unicodedata.normalize("NFC", s))
            after = df[col].nunique()
            if before != after:
                print(f"  [fixed] '{col}': unicode/whitespace normalization merged "
                      f"{before} -> {after} unique values (likely duplicate categories).")
            else:
                print(f"  [ok] '{col}': {after} unique values, no normalization needed.")

    # --- 2. Drop exact duplicate rows ---
    n_before = len(df)
    df = df.drop_duplicates()
    if len(df) < n_before:
        print(f"  [fixed] dropped {n_before - len(df)} exact duplicate row(s).")
    else:
        print("  [ok] no exact duplicate rows found.")

    # --- 3. Duplicate (artist, raga, filepath) combos -- same song counted twice ---
    if {"artist", "raga", "filepath"}.issubset(df.columns):
        dupe_mask = df.duplicated(subset=["artist", "raga", "filepath"], keep="first")
        if dupe_mask.any():
            print(f"  [fixed] dropped {dupe_mask.sum()} row(s) with a repeated "
                  f"(artist, raga, filepath) combination.")
            df = df[~dupe_mask]
        else:
            print("  [ok] no repeated (artist, raga, filepath) combinations.")

    # --- 4. Missing values in feature columns ---
    n_missing = df[feature_cols].isna().sum().sum()
    if n_missing > 0:
        rows_with_na = df[feature_cols].isna().any(axis=1).sum()
        print(f"  [fixed] found {n_missing} missing feature value(s) across "
              f"{rows_with_na} row(s) -- dropping those rows (too few rows here "
              f"to safely impute).")
        df = df.dropna(subset=feature_cols)
    else:
        print("  [ok] no missing values in feature columns.")

    # --- 5. Infinite values in feature columns ---
    inf_mask = np.isinf(df[feature_cols].values).any(axis=1)
    if inf_mask.any():
        print(f"  [fixed] found {inf_mask.sum()} row(s) with infinite feature "
              f"values -- dropping them.")
        df = df[~inf_mask]
    else:
        print("  [ok] no infinite values in feature columns.")

    # --- 6. Sanity check: are chroma means within a sane [0, 1] range? ---
    # (Unlike PCD, MFCC has no natural bound, but chroma-CQT energy should
    # sit in roughly [0, 1] per bin -- a useful red flag for an upstream bug.)
    chroma_mean_cols = [c for c in feature_cols if c.startswith("chroma_") and c.endswith("_mean")]
    if chroma_mean_cols:
        vals = df[chroma_mean_cols].values
        out_of_range = ((vals < -1e-6) | (vals > 1 + 1e-6)).any(axis=1).sum()
        if out_of_range > 0:
            print(f"  [warning] {out_of_range} row(s) have chroma mean values outside "
                  f"the expected [0, 1] range. This may indicate a bug upstream in "
                  f"feature extraction -- not auto-fixed, please review manually.")
        else:
            print("  [ok] chroma mean values are within the expected [0, 1] range.")

    # --- 7. Zero-variance (constant) feature columns ---
    variances = df[feature_cols].var()
    zero_var_cols = variances[variances == 0].index.tolist()
    if zero_var_cols:
        shown = zero_var_cols[:5]
        more = "..." if len(zero_var_cols) > 5 else ""
        print(f"  [fixed] dropping {len(zero_var_cols)} zero-variance (constant) "
              f"feature column(s): {shown}{more}")
        df = df.drop(columns=zero_var_cols)
    else:
        print("  [ok] no zero-variance feature columns.")

    # --- 8. Class balance check ---
    class_counts = df["raga"].value_counts()
    if class_counts.min() != class_counts.max():
        print(f"  [warning] class counts are NOT balanced: "
              f"min={class_counts.min()} ({class_counts.idxmin()}), "
              f"max={class_counts.max()} ({class_counts.idxmax()}).")
    else:
        print(f"  [ok] all {len(class_counts)} raga classes have exactly "
              f"{class_counts.iloc[0]} songs each -- balanced.")

    # --- 9. Artist diversity per raga (needed for the unseen-evaluation split) ---
    artist_counts = df.groupby("raga")["artist"].nunique()
    single_artist_ragas = artist_counts[artist_counts < 2].index.tolist()
    if single_artist_ragas:
        print(f"  [warning] these raga(s) have only 1 unique artist, so no "
              f"'unseen' test split is possible for them: {single_artist_ragas}")
    else:
        print(f"  [ok] every raga has at least 2 unique artists "
              f"(min={artist_counts.min()}) -- unseen split is possible for all classes.")

    n_final_feats = len(get_feature_columns(df, "both"))
    print(f"\nAfter preprocessing: {len(df)} rows, {df['raga'].nunique()} classes, "
          f"{n_final_feats} feature columns.")
    return df


# ===========================================================================
# 2. SPLIT OUT FEATURE COLUMNS FOR A GIVEN FEATURE SET
# ===========================================================================
def get_feature_columns(df: pd.DataFrame, feature_set: str = "both") -> list:
    """Return the mfcc_*/chroma_* columns belonging to the requested feature set.

    feature_set: "mfcc"   -> mfcc_* columns only (incl. mfcc_delta_*)  -- timbre
                 "chroma" -> chroma_* columns only                     -- pitch-class
                 "both"   -> mfcc_* + chroma_* columns together
    """
    mfcc_cols = [c for c in df.columns if c.startswith("mfcc_")]
    chroma_cols = [c for c in df.columns if c.startswith("chroma_")]

    if feature_set == "mfcc":
        return mfcc_cols
    elif feature_set == "chroma":
        return chroma_cols
    elif feature_set == "both":
        return mfcc_cols + chroma_cols
    else:
        raise ValueError(f"Unknown feature_set '{feature_set}', expected mfcc/chroma/both")


def prepare_data(df: pd.DataFrame, feature_set: str = "both"):
    feature_cols = get_feature_columns(df, feature_set)
    X = df[feature_cols].values
    y_raw = df["raga"].values
    groups = df["artist"].values   # used later for the artist-independent "unseen" split

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)

    print(f"Feature matrix ({feature_set}): {X.shape}  ({len(feature_cols)} features)")
    print(f"Classes ({len(label_encoder.classes_)}): {list(label_encoder.classes_)}")

    return X, y, groups, label_encoder


# ===========================================================================
# 3. THE MODELS
# ===========================================================================
# With only ~10 samples/class, XGBoost and a plain fixed-hyperparameter SVM
# both tend to overfit or land on the wrong C/gamma for such a small
# dataset. This swaps in models better suited to that regime:
#
#   - "SVM (grid)":    same RBF SVM, but C/gamma/kernel are chosen via a
#                       grid search *nested inside* every CV fold (the grid
#                       search only ever sees that fold's training data, so
#                       there's no leakage into the fold's test data).
#   - "KNN (cosine)":  chroma vectors describe a *shape* (relative energy
#                       across pitch classes), not an absolute magnitude --
#                       cosine distance compares direction/shape and ignores
#                       overall scale, which usually suits that better than
#                       KNN's default Euclidean distance.
#   - "RandomForest":  an ensemble of many small, high-variance trees --
#                       tends to be more robust than a single model when
#                       there isn't much data to fit complex boundaries with.
#   - "GaussianNB":    a simple Gaussian likelihood model per class, in the
#                       same spirit as the reference paper's MVN classifier
#                       (their second-best method, ~96% CV in the paper) --
#                       a good low-variance baseline for a small dataset.
#   - "XGBoost":       kept for comparison against the above.
def build_models():
    svm_param_grid = {
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", 0.1, 0.01, 0.001],
        "kernel": ["rbf", "linear"],
    }
    # cv=3 for the *inner* grid search (separate from the outer 10-fold CV
    # in run_cv_evaluation) -- kept small since each outer fold already only
    # has ~9 samples/class to search over.
    svm_grid = GridSearchCV(
        SVC(random_state=RANDOM_STATE),
        param_grid=svm_param_grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
    )

    return {
        "SVM (grid)": svm_grid,
        "KNN (cosine)": KNeighborsClassifier(n_neighbors=5, metric="cosine"),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, max_depth=None, random_state=RANDOM_STATE
        ),
        "GaussianNB": GaussianNB(),
        "XGBoost": XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.1,
            eval_metric="mlogloss", random_state=RANDOM_STATE,
        ),
    }


# ===========================================================================
# 4. PART 1: 10-FOLD CROSS-VALIDATION
# ===========================================================================
def run_cv_evaluation(X, y, n_folds=N_CV_FOLDS):
    """
    Standard 10-fold stratified CV. Features are standardized (mean 0, std 1)
    inside each fold's training data only, via a Pipeline, so there's no
    leakage from test fold into the scaling.
    """
    from sklearn.pipeline import Pipeline

    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    results = {}

    for name, model in build_models().items():
        pipeline = Pipeline([("scaler", StandardScaler()), ("clf", model)])
        scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy")
        results[name] = scores
        print(f"{name:10s}: mean accuracy = {scores.mean()*100:.2f}%  "
              f"(std = {scores.std()*100:.2f}%)  folds = {np.round(scores*100, 1)}")

    return results


# ===========================================================================
# 5. PART 2: "UNSEEN" EVALUATION -- artist-independent train/test split
# ===========================================================================
def artist_independent_split(df: pd.DataFrame, test_size=UNSEEN_TEST_SIZE, random_state=RANDOM_STATE,
                              max_attempts=50):
    """
    Splits by ARTIST GLOBALLY, not per-raga -- every artist ends up entirely
    in train OR entirely in test, across ALL the ragas they performed. This
    guarantees zero leakage even when the same artist (e.g. a famous
    vocalist) appears across multiple different ragas in the dataset, which
    a naive per-raga split would miss.

    Because some famous artists span many ragas, a single random split can
    accidentally leave a raga with zero test songs (if all its artists
    happen to land in train). To reduce that risk without ever risking
    leakage, this tries several random seeds and keeps whichever one covers
    the most raga classes in the test set -- but every attempt is leakage-free
    by construction (GroupShuffleSplit never splits one group across both
    sides), so this search only affects *class coverage*, never correctness.
    """
    all_ragas = set(df["raga"].unique())
    best_split = None
    best_coverage = -1

    for attempt in range(max_attempts):
        seed = random_state + attempt
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        train_idx, test_idx = next(gss.split(df, groups=df["artist"]))
        train_df = df.iloc[train_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)

        coverage = test_df["raga"].nunique()
        if coverage > best_coverage:
            best_coverage = coverage
            best_split = (train_df, test_df)
        if coverage == len(all_ragas):
            break  # every raga is represented in the test set -- good enough, stop searching

    train_df, test_df = best_split

    overlap = set(train_df["artist"]) & set(test_df["artist"])
    assert not overlap, f"Artist leakage between train/test: {overlap}"

    missing_ragas = all_ragas - set(test_df["raga"].unique())
    if missing_ragas:
        print(f"  [warning] {len(missing_ragas)} raga(s) have NO songs in the unseen "
              f"test set after {max_attempts} attempts (all their artists landed in "
              f"training): {sorted(missing_ragas)}")
    else:
        print(f"  [ok] every raga has at least one song in the unseen test set "
              f"(found after {attempt + 1} attempt(s)).")

    return train_df, test_df


def run_unseen_evaluation(df: pd.DataFrame, label_encoder: LabelEncoder, feature_set: str = "both",
                           verbose: bool = True):
    train_df, test_df = artist_independent_split(df)
    feature_cols = get_feature_columns(df, feature_set)

    X_train = train_df[feature_cols].values
    X_test = test_df[feature_cols].values
    y_train = label_encoder.transform(train_df["raga"].values)
    y_test = label_encoder.transform(test_df["raga"].values)

    print(f"Train: {len(X_train)} songs | Unseen test: {len(X_test)} songs "
          f"(from artists not in training)")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    results = {}
    for name, model in build_models().items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)
        results[name] = acc
        print(f"\n{name}: unseen accuracy = {acc*100:.2f}%")
        if verbose:
            labels_present = np.unique(np.concatenate([y_test, y_pred]))
            print(classification_report(
                y_test, y_pred,
                labels=labels_present,
                target_names=label_encoder.inverse_transform(labels_present),
                zero_division=0,
            ))

    return results


# ===========================================================================
# 6. RUN ALL THREE FEATURE SETS (MFCC, CHROMA, BOTH) AND COMPARE
# ===========================================================================
def run_feature_set(df: pd.DataFrame, feature_set: str, verbose: bool = True):
    """Run the full CV + unseen evaluation pipeline for one feature set."""
    print(f"\n{'#'*60}\n# FEATURE SET: {feature_set.upper()}\n{'#'*60}")

    X, y, groups, label_encoder = prepare_data(df, feature_set)

    print(f"\n{'='*60}\n10-FOLD CROSS-VALIDATION ({feature_set})\n{'='*60}")
    cv_results = run_cv_evaluation(X, y)

    print(f"\n{'='*60}\nUNSEEN EVALUATION ({feature_set})\n{'='*60}")
    unseen_results = run_unseen_evaluation(df, label_encoder, feature_set, verbose=verbose)

    return cv_results, unseen_results


def main(data_dir=DATA_DIR, feature_sets=FEATURE_SETS, verbose=True):
    print("Loading data...")
    df = load_all_ragas(data_dir)
    df = preprocess_data(df)

    all_results = {}   # {feature_set: (cv_results, unseen_results)}
    for feature_set in feature_sets:
        all_results[feature_set] = run_feature_set(df, feature_set, verbose=verbose)

    # ------------------------------------------------------------------
    # Combined summary -- Feature Set x Model x CV / Unseen.
    # ------------------------------------------------------------------
    print(f"\n{'='*60}\nSUMMARY -- ALL FEATURE SETS\n{'='*60}")
    header = f"{'Feature Set':12s} | {'Model':10s} | {'10-fold CV':>12s} | {'Unseen':>10s}"
    print(header)
    print("-" * len(header))
    for feature_set in feature_sets:
        cv_results, unseen_results = all_results[feature_set]
        for name in build_models().keys():
            cv_acc = cv_results[name].mean() * 100
            unseen_acc = unseen_results[name] * 100
            print(f"{feature_set:12s} | {name:10s} | {cv_acc:11.2f}% | {unseen_acc:9.2f}%")
        print("-" * len(header))

    # Mean-across-models accuracy per feature set, for a quick eyeball
    # comparison of MFCC vs chroma vs both.
    print(f"\n{'-'*40}\nMean accuracy across all 3 models, per feature set\n{'-'*40}")
    print(f"{'Feature Set':12s} | {'10-fold CV':>12s} | {'Unseen':>10s}")
    for feature_set in feature_sets:
        cv_results, unseen_results = all_results[feature_set]
        mean_cv = np.mean([s.mean() for s in cv_results.values()]) * 100
        mean_unseen = np.mean(list(unseen_results.values())) * 100
        print(f"{feature_set:12s} | {mean_cv:11.2f}% | {mean_unseen:9.2f}%")

    return all_results


In [2]:
all_results = main(data_dir="/kaggle/input/datasets/mishbhaul/mfcc-chroma-hcm-dataset")  

Loading data...
  loaded Kdr.csv: 10 rows, raga(s)=['Kēdār']
  loaded ahira_bhairav.csv: 10 rows, raga(s)=['Ahira bhairav']
  loaded bhairav.csv: 10 rows, raga(s)=['Bhairav']
  loaded bhg.csv: 10 rows, raga(s)=['Ābhōgī']
  loaded madhukauns.csv: 10 rows, raga(s)=['Madhukauns']
  loaded madhuvanti.csv: 10 rows, raga(s)=['Madhuvanti']
  loaded mrbihg.csv: 10 rows, raga(s)=['Mārūbihāg']
  loaded mrv.csv: 10 rows, raga(s)=['Mārvā']
  loaded r.csv: 10 rows, raga(s)=['Śrī']
  loaded rgr.csv: 10 rows, raga(s)=['Rāgēśrī']

Combined dataset: 100 rows, 10 raga classes

PREPROCESSING
  [ok] 'raga': 10 unique values, no normalization needed.
  [ok] 'artist': 33 unique values, no normalization needed.
  [ok] no exact duplicate rows found.
  [ok] no repeated (artist, raga, filepath) combinations.
  [ok] no missing values in feature columns.
  [ok] no infinite values in feature columns.
  [ok] chroma mean values are within the expected [0, 1] range.
  [ok] no zero-variance feature columns.
  [ok] all